<a href="https://colab.research.google.com/github/personallypetra/Grand_Challenge-/blob/Petra/PetraAccenture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
%%capture
!rm -rf Grand_Challenge
!git clone https://github.com/personallypetra/Grand_Challenge-.git
%cd Grand_Challenge-
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sqlalchemy import create_engine, inspect

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    auc
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

In [18]:
import os
print(os.listdir("data_petra"))

['Export-DownloadCenterFile-20260307-205601.xlsx', 'Export-DownloadCenterFile-20260307-163501.xlsx', 'Export-DownloadCenterFile-20260307-205628.xlsx', 'Export-DownloadCenterFile-20260307-205608.xlsx', 'Export-DownloadCenterFile-20260307-205623.xlsx', 'Export-DownloadCenterFile-20260307-205618.xlsx', 'Export-DownloadCenterFile-20260307-162932.xlsx', 'Export-DownloadCenterFile-20260307-205613.xlsx', 'Export-DownloadCenterFile-20260307-163528.xlsx']


In [19]:
df1 = pd.read_excel("data_petra/Export-DownloadCenterFile-20260307-205601.xlsx")
df2 = pd.read_excel("data_petra/Export-DownloadCenterFile-20260307-205608.xlsx")
df3 = pd.read_excel("data_petra/Export-DownloadCenterFile-20260307-205613.xlsx")
df4 = pd.read_excel("data_petra/Export-DownloadCenterFile-20260307-205618.xlsx")
df5 = pd.read_excel("data_petra/Export-DownloadCenterFile-20260307-205623.xlsx")
df6 = pd.read_excel("data_petra/Export-DownloadCenterFile-20260307-205628.xlsx")

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default sty

In [20]:
df = pd.concat([df1, df2, df3, df4, df5, df6], ignore_index  = True)
df.to_excel("Total_Load_Full.xlsx", index = False)
df.head()

,Date,Total Load [MW],Forecast Total Load [MW],Bidding Zone
0,2026-03-06 23:45:00,700.942,692.646,Calabria
1,2026-03-06 23:45:00,2157.437,2131.902,Centre-North
2,2026-03-06 23:45:00,5216.721,5154.979,Centre-South
3,2026-03-06 23:45:00,18011.345,17798.173,North
4,2026-03-06 23:45:00,760.850,751.845,Sardinia


The available database measures total load and forecast Total Load each 15 minutes. We are now creating new database that is going to contain same informations on daily basis.

In [21]:
df = df[~df["Date"].astype(str).str.contains("Applied filters", na = False)]
df["Date"] = pd.to_datetime(df["Date"], dayfirst = True)
df["Date"] = df["Date"].dt.floor("D")

TotalLoadDaily = df.groupby(["Date", "Bidding Zone"], as_index = False).agg(
    {"Total Load [MW]":"sum",
     "Forecast Total Load [MW]": "sum"
                                                                             })
TotalLoadDaily.to_excel("Total_Load_Daily.xlsx", index = False)
TotalLoadDaily.head(9)


,Date,Bidding Zone,Total Load [MW],Forecast Total Load [MW]
0,2021-03-01,Calabria,64020.862,63488.833
1,2021-03-01,Centre-North,283967.215,281751.575
2,2021-03-01,Centre-South,576870.367,572046.452
3,2021-03-01,Italy,3510784.002,3479473.005
4,2021-03-01,North,2031942.282,2013133.832
5,2021-03-01,Sardinia,95270.554,94358.747
6,2021-03-01,Sicily,206078.953,204352.898
7,2021-03-01,South,252633.769,250340.668
8,2021-03-02,Calabria,63032.749,63038.980


In [22]:
TotalLoadDaily["Bidding Zone"].unique()

array(['Calabria', 'Centre-North', 'Centre-South', 'Italy', 'North',
       'Sardinia', 'Sicily', 'South'], dtype=object)

In [23]:
print(TotalLoadDaily["Date"].head(1))
print(TotalLoadDaily["Date"].iloc[-1])

0   2021-03-01
Name: Date, dtype: datetime64[ns]
2026-03-06 00:00:00


Now we have one database "TotalLoadDaily" that contains both predicted and actual total usage of electricity grouped by regions from 1st March 2021 till 6th March 2026. Moreover, we have a daily usage for the whole Italy.